# Geometry certification, expansion refinement, and pivot amplitude

This prototype covers the diagnostic and hyperparameter-geometry pieces of the
timing-coordinate-charts feature:

1. **Fixed-expansion refinement** — move the linearization expansion to the
   conditional timing MPE (a *marginal* objective).
2. **Off-zero geometry certification** — probe the exact NumPyro joint target at
   the deterministic `2K+9` points across box hyper-probes; report center
   interiority, exact-vs-local residual remainder, the `xi` gradient/Hessian, and
   the conditional-identity spread. Nothing here ever runs inside NUTS.
   **Section 2b is the headline result**: the certifier catches a ~million-fold
   off-mode geometry defect (spin parameters on wide uniform charts — the same
   `F0` axis that broke the earlier decentering run), and a single modeling knob,
   `identically_linear=`, turns it into a clean pass.
3. **Pivot-amplitude power laws** — sample the red-noise amplitude at a
   sensitivity-weighted pivot frequency to decorrelate amplitude and slope.

Simulated data; run in the devcontainer.


In [1]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

import tempfile
from pathlib import Path

import numpy as np

import discovery as ds
from discovery import transport as dst
from metapulsar import create_metapulsar
from metapulsar.sandbox_tempo2 import configure_logging
from nltiming import (
    NonLinearTimingModel, TimingInference,
    box_hyper_probe_points, certify_joint_geometry, transport_center_report,
    write_geometry_report, read_geometry_report, refine_timing_expansion,
)
from nltiming.sampling import numpyro as N

ds.config(kernels="metamath")
configure_logging(level="WARNING")
workdir = Path(tempfile.mkdtemp(prefix="nlt_geom_"))


In [2]:
import pint.config
from pint.models import get_model
from pint.simulation import make_fake_toas_uniform

np.random.seed(7)
model = get_model(pint.config.examplefile("NGC6440E.par"))
toas = make_fake_toas_uniform(
    startMJD=53400, endMJD=56000, ntoas=150, model=model, obs="gbt",
    error=1.0, add_noise=True,
)
(workdir / "d.par").write_text(model.as_parfile())
toas.write_TOA_file(str(workdir / "d.tim"), format="tempo2")
mp = create_metapulsar(
    {"demo": [{"par": str(workdir / "d.par"), "tim": str(workdir / "d.tim"),
               "timing_package": "pint"}]},
    use_pulse_numbers="no",
)

ctx = NonLinearTimingModel(
    engines="jug", inference=TimingInference.sample_all(), name="timing"
).for_pulsar(mp)
nd = {f"{mp.name}_efac": 1.0, f"{mp.name}_log10_t2equad": -8.0}
reference = dst.reference_noise_frozen(
    ds.makenoise_measurement_simple(mp, nd), nd, description="WN MPE")
center = {f"{mp.name}_rednoise_log10_A": -14.0, f"{mp.name}_rednoise_gamma": 3.0}
print("sampled timing axes:", len(ctx.sampled))


[!] Unrecognized par file parameters (ignored by JUG): DMDATA, NE_SW1
[FITTER] noise_config was None, auto-detecting from par file
sampled timing axes: 8


## 1. Refine the fixed expansion at the conditional MPE

`conditional_timing_potential` takes a **marginal** likelihood (`joint=False`, red
noise analytically integrated, timing via delay keys) — not the residual-form
joint `clogL`. `refine_timing_expansion` runs SciPy L-BFGS-B over the jitted
value-and-gradient and returns a context re-linearized at the refined expansion
(or the input context if it did not converge). A well-fit par file is already at
the timing MPE, so `delta = 0` is often already optimal.


In [3]:
psl_marginal = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx.discovery_signals(joint=False),
])
objective = N.conditional_timing_potential(psl_marginal, ctx, fixed={**nd, **center})
refined = refine_timing_expansion(ctx, negative_log_target_z=objective)
ctx_cert = refined.context
z0 = np.asarray(ctx.linearization.sampled_z_expansion)
z1 = np.asarray(ctx_cert.linearization.sampled_z_expansion)
print("converged:", refined.converged, " |d z_e|:", float(np.linalg.norm(z1 - z0)))


converged: False  |d z_e|: 0.0


## 2. Certify the joint target geometry

Build the joint model on the (refined) context and certify it at the box hyper
probes. `passed` is advisory only — it never controls NUTS. On real data a
`passed=False` with a large residual remainder or an eigenvalue far from 1 is a
genuine finding of intra-posterior curvature, not a bug.


In [4]:
psl_joint = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_cert.discovery_signals(joint=True),
])
jm = N.joint_model(psl_joint, ctx_cert, reference_noise=reference, fixed=nd)

bounds = {f"{mp.name}_rednoise_log10_A": (-18.0, -11.0),
          f"{mp.name}_rednoise_gamma": (0.0, 7.0)}
hyper_points = box_hyper_probe_points(center, bounds)[:3]
report = certify_joint_geometry(jm, ctx_cert, hyper_points=hyper_points)
print("passed :", report.passed)
print("rms    :", round(report.max_residual_remainder_rms, 4),
      " max std/TOA:", round(report.max_residual_remainder_standardized_toa, 3))
print("xi grad:", round(report.max_xi_gradient_inf_norm, 4),
      " H eig  :", round(report.xi_hessian_eigen_min, 3), "..",
      round(report.xi_hessian_eigen_max, 3))
print("id spread:", round(report.max_conditional_identity_spread, 4))
for f in report.failures:
    print("  FAIL:", f)


passed : False
rms    : 625.0731  max std/TOA: 1474.542
xi grad: 543885.4311  H eig  : -4105.997 .. 2910467.91
id spread: 29301842.3738
  FAIL: residual_remainder_rms=625 > 0.1
  FAIL: residual_remainder_max_standardized_toa=1.47e+03 > 1.0
  FAIL: xi_gradient_inf_norm=5.44e+05 > 0.2
  FAIL: xi_hessian_eigen_min=-4.11e+03 < 0.5
  FAIL: xi_hessian_eigen_max=2.91e+06 > 2.0
  FAIL: xi_eta_cross_operator_norm=6.77e+05 > 0.25
  FAIL: conditional_identity_spread=2.93e+07 > 0.1


## 2b. The one knob that matters most: `identically_linear`

The certification above **failed catastrophically** — a residual RMS in the
hundreds, a Hessian eigenvalue near 10⁶, and even a *negative* eigenvalue. That
is **not** a solver glitch: it is the certifier correctly reporting that this
sampling geometry is broken away from the mode.

**Why?** Look back at the chart summary. `F0` and `F1` are *physically* linear in
pulse phase, but nltiming's conservative, engine-independent fallback registry
does **not** certify them identically linear — so they were sampled through a
**wide uniform `prior_pit` chart**. A unit-`xi` probe then reaches deep into that
prior's tails, where spin sensitivity is enormous and the exact timing residual
explodes. (Notice in the transport-center report that `F1` already sits at
`center_z ≈ 0.35`, off the expansion — the wide uniform chart has real curvature
there.) **This is exactly the ≈2.5-width `F0` axis that broke the *previous*
decentering run** — the Stage 0 J1640 forensic, reproduced here on toy data.

The fix is a single argument. `F0`/`F1`/`RAJ`/`DECJ` really are linear in their
delta, so we may **declare** them identically linear. That swaps their `prior_pit`
charts for globally-affine `affine_normal` charts (a Gaussian delta prior with a
constant `dδ/dz`), and the off-mode geometry collapses by ~6 orders of magnitude:


In [ ]:
# `identically_linear=` is AUTHORITATIVE: an explicit list becomes *exactly* the
# certified set, dropping any fallback/engine candidates it omits. So to ADD the
# physically-linear spin/position axes we UNION them with the auto-derived set
# (which already certifies DM/DM1/DM2/Offset) -- otherwise those would silently
# fall back to prior_pit.
declared = sorted(set(ctx_cert.identically_linear) | {"F0", "F1", "RAJ", "DECJ"})
ntm_lin = NonLinearTimingModel(
    engines="jug",
    inference=TimingInference.sample_all(),
    identically_linear=declared,
    name="timing",
)
ctx_lin = ntm_lin.for_pulsar(mp)

# Only F0/F1/RAJ/DECJ flip; the DM family stays affine_normal:
charts_default = {d["name"]: d["chart"] for d in ctx_cert.chart_summary()}
charts_lin = {d["name"]: d["chart"] for d in ctx_lin.chart_summary()}
print("axis    default chart   ->  declared chart")
for name in ("F0", "F1", "RAJ", "DECJ", "DM", "DM1"):
    flag = "  <-- flipped" if charts_default[name] != charts_lin[name] else ""
    print(f"  {name:5s} {charts_default[name]:14s} ->  {charts_lin[name]:14s}{flag}")

# Rebuild the joint model and certify at the SAME hyper probes.
psl_lin = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_lin.discovery_signals(joint=True),
])
jm_lin = N.joint_model(psl_lin, ctx_lin, reference_noise=reference, fixed=nd)
report_lin = certify_joint_geometry(jm_lin, ctx_lin, hyper_points=hyper_points)


def _row(label, r):
    return (f"  {label:26s}  rms={r.max_residual_remainder_rms:11.4g}"
            f"  max_std/TOA={r.max_residual_remainder_standardized_toa:11.4g}"
            f"  xi_grad={r.max_xi_gradient_inf_norm:11.4g}"
            f"  H_eig=[{r.xi_hessian_eigen_min:10.4g}, {r.xi_hessian_eigen_max:11.4g}]"
            f"  id_spread={r.max_conditional_identity_spread:11.4g}")


print("\nSAME data, SAME probes -- the only change is the chart on F0/F1/RAJ/DECJ:")
print(_row("F0/F1 uniform (default)", report))
print(_row("F0/F1 identically_linear", report_lin))
shrink = report.xi_hessian_eigen_max / report_lin.xi_hessian_eigen_max
print(f"\n-> Hessian eigenvalue max shrank by ~{shrink:,.0f}x; the negative "
      f"eigenvalue (a saddle off the mode) is gone, and the Hessian is ~identity.")
print(f"   passed: default={report.passed} -> declared={report_lin.passed}   "
      f"(residual failures: {len(report.failures)} -> {len(report_lin.failures)})")


**This is the whole point of the feature.** The certifier is a cheap, exact,
*off-mode* geometry check that runs **before** any sampling. A `passed=False`
with runaway metrics is a **design signal, not a solver failure**: here it caught
a ~million-fold geometry defect that NUTS would have papered over near the mode
and then blown up on in the tails — precisely the failure mode of the earlier
decentering implementation, and the same `F0` axis flagged in the Stage 0 J1640
forensic.

The remedy is a **modeling decision** (`identically_linear=`), *not* loosening a
threshold or raising the tree depth. For this isolated pulsar every timing
parameter is exactly linear, so declaring them collapses the geometry to a **clean
pass** — Hessian ≈ identity, remainder ≈ 0. On richer real-data models (free red
noise, binary orbits, a white-noise-only reference) the report may *still* not
fully pass even after this fix — as on J1640, where the WN reference cannot
precondition the timing↔red-noise cross-curvature. That residual is itself
actionable: it names the next thing to build (a red-noise-aware reference, or the
block bijector), rather than a number quietly forced under a bar.


### Transport-center report and standalone products

`transport_center_report` distinguishes an `affine_normal` axis (interior for any
finite center) from a `prior_pit` axis (interior only within the PIT limit). The
report writes to a standalone JSON + NPZ pair that round-trips and verifies its
own digests — independent of the run manifest.


In [5]:
axes = transport_center_report(ctx_cert, jm.transport, center)
for a in axes[:5]:
    print(f"{a.name:12s} chart={a.chart:14s} center_z={a.center_z:+.3f} "
          f"interior={a.interior} chart_ratio={a.local_chart_ratio}")

stem = workdir / "geometry_report"
json_path, npz_path = write_geometry_report(report, stem, overwrite=True)
reloaded = read_geometry_report(stem)
print("\nwrote", json_path.name, "+", npz_path.name,
      "| digest-verified roundtrip:", reloaded.model_fingerprint == report.model_fingerprint)


RAJ          chart=prior_pit      center_z=+0.000 interior=True chart_ratio=0.9999999958667124
DECJ         chart=prior_pit      center_z=-0.000 interior=True chart_ratio=0.9999999999688715
F0           chart=prior_pit      center_z=-0.045 interior=True chart_ratio=0.9989675415248926
F1           chart=prior_pit      center_z=+0.353 interior=True chart_ratio=0.939457008589444
DM           chart=affine_normal  center_z=-0.016 interior=True chart_ratio=None

wrote geometry_report.json + geometry_report.npz | digest-verified roundtrip: True


## 3. Pivot-amplitude red noise

Sampling `log10_A` at `f = 1/yr` correlates amplitude and slope. Sampling
`log10_A_pivot` at a sensitivity-weighted pivot frequency decorrelates them via
the affine, unit-Jacobian map
`log10_A_ref = log10_A_pivot + 1/2 gamma log10(f_pivot / f_ref)`.

The pivot frequency comes from the fixed reference-noise metric: weights
`w_j = tr(F_j^T N0^-1 F_j)`, then a sensitivity-weighted geometric mean of `f_j`.


In [6]:
NCOMP = 10
f, df, fmat = ds.fourierbasis(mp, NCOMP)
weights = ds.fourier_sensitivity_weights(fmat, dst.reference_noise(mp))
f_pivot = ds.sensitivity_weighted_pivot_frequency(np.asarray(f)[0::2], weights)
print(f"sensitivity-weighted pivot: {f_pivot:.3e} Hz  (1/yr = {ds.const.fyr:.3e} Hz)")

# Same spectrum, reparameterized: log10_A_pivot decoded to the reference amplitude
# reproduces the classic power law exactly.
param = ds.PowerLawParameterization(slope_pivot_frequency=f_pivot)
pl_pivot = ds.make_powerlaw_pivot(f_pivot=f_pivot, parameterization=param)
pl_ref = ds.make_powerlaw()
log10_A_pivot, gamma = -14.3, 3.7
log10_A_ref = ds.reference_log10_amplitude(
    log10_A_pivot, gamma, f_pivot=f_pivot, parameterization=param)
ff = np.array([1e-9, 3e-9, 1e-8])
dff = np.full_like(ff, 1e-9)
same = np.allclose(pl_pivot(ff, dff, log10_A_pivot, gamma),
                   pl_ref(ff, dff, log10_A_ref, gamma), rtol=1e-12)
print("pivot spectrum == reference spectrum:", same,
      f"| log10_A(1/yr) = {log10_A_ref:.4f}")


sensitivity-weighted pivot: 2.016e-08 Hz  (1/yr = 3.169e-08 Hz)
pivot spectrum == reference spectrum: True | log10_A(1/yr) = -14.6634


### Wiring the pivot PSD into a real red-noise GP

`make_powerlaw_pivot` is a drop-in PSD for `makegp_fourier`: the resulting GP's
sampled amplitude is named `..._rednoise_pivot_log10_A_pivot` (defined at
`f_pivot`) instead of `..._log10_A` (defined at `1/yr`) — an unambiguous public
name. Decode back to `1/yr` for display with `reference_log10_amplitude`.

(When you feed a pivot GP to `N.joint_model`/`N.nuts`, supply explicit
`priors=` for the `log10_A_pivot`/`gamma` sites — the default 1/yr amplitude
prior does not auto-resolve for the pivot suffix.)


In [7]:
rn_pivot = ds.makegp_fourier(mp, pl_pivot, NCOMP, name="rednoise_pivot")
psl_pivot = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    rn_pivot,
])
pivot_params = [p for p in psl_pivot.logL.params if "rednoise_pivot" in p]
print("pivot RN sampled params:", pivot_params)
print("-> amplitude is sampled at f_pivot as '...rednoise_pivot_log10_A_pivot';")
print("   decode to 1/yr with reference_log10_amplitude(..., f_pivot=f_pivot).")


pivot RN sampled params: ['J1748-2021_rednoise_pivot_gamma', 'J1748-2021_rednoise_pivot_log10_A_pivot']
-> amplitude is sampled at f_pivot as '...rednoise_pivot_log10_A_pivot';
   decode to 1/yr with reference_log10_amplitude(..., f_pivot=f_pivot).
